In [ ]:
!pip install openai python-dotenv

In [ ]:
#CELDA 0
import subprocess, sys

# Verifica dependencias
subprocess.run([sys.executable, "-m", "pip", "install", 
                "python-dotenv", "openai", "-q"])

from dotenv import load_dotenv
import os
load_dotenv()

token = os.getenv("GITHUB_TOKEN")
url   = os.getenv("OPENAI_BASE_URL")
print(f"Token:   {'✅ OK' if token else '❌ Renovar en github.com/settings/tokens'}")
print(f"API URL: {'✅ ' + url if url else '❌ Falta en .env'}")

In [ ]:
# CELDA 1
import logging
import time
import random
import json
from datetime import datetime
from dataclasses import dataclass, field
from typing import List
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

# Verificar que las variables del .env se cargan correctamente
github_token = os.getenv("GITHUB_TOKEN")
base_url = os.getenv("OPENAI_BASE_URL")

print(f"GITHUB_TOKEN cargado: {'✅' if github_token else '❌ NO encontrado'}")
print(f"OPENAI_BASE_URL: {base_url if base_url else '❌ NO encontrado'}")

# --- Configuracion de logging estructurado con timestamps ---
formato_log = logging.Formatter(
    fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

manejador_consola = logging.StreamHandler()
manejador_consola.setFormatter(formato_log)

logger = logging.getLogger("observabilidad_vulcanizacion")
logger.setLevel(logging.DEBUG)
if not logger.handlers:
    logger.addHandler(manejador_consola)

logger.info("Sistema de observabilidad iniciado correctamente")

In [ ]:
# CELDA 2
# --- Recolector de metricas ---
@dataclass
class RegistroMetrica:
    """Registro individual de una interaccion del sistema de vulcanizacion."""
    timestamp: str
    tiempo_respuesta_ms: float
    tokens_entrada: int
    tokens_salida: int
    exitoso: bool
    modelo: str
    consulta_tipo: str  # tipo de servicio consultado: parche, cambio, alineacion, etc.

In [ ]:
# CELDA 3
class RecolectorMetricas:
    """Recolecta y resume metricas de rendimiento del agente de vulcanizacion."""

    def __init__(self):
        self.registros: List[RegistroMetrica] = []

    def registrar(self, tiempo_ms: float, tokens_in: int, tokens_out: int,
                  exitoso: bool, consulta_tipo: str, modelo: str = "openai/gpt-4.1"):
        registro = RegistroMetrica(
            timestamp=datetime.now().isoformat(),
            tiempo_respuesta_ms=round(tiempo_ms, 2),
            tokens_entrada=tokens_in,
            tokens_salida=tokens_out,
            exitoso=exitoso,
            modelo=modelo,
            consulta_tipo=consulta_tipo,
        )
        self.registros.append(registro)

    def resumen(self) -> dict:
        """Devuelve un resumen con estadisticas agregadas."""
        if not self.registros:
            return {"total_peticiones": 0}

        tiempos = [r.tiempo_respuesta_ms for r in self.registros]
        total_tokens = sum(r.tokens_entrada + r.tokens_salida for r in self.registros)
        errores = sum(1 for r in self.registros if not r.exitoso)

        # Conteo por tipo de consulta
        tipos = {}
        for r in self.registros:
            tipos[r.consulta_tipo] = tipos.get(r.consulta_tipo, 0) + 1

        return {
            "total_peticiones": len(self.registros),
            "tiempo_promedio_ms": round(sum(tiempos) / len(tiempos), 2),
            "tiempo_maximo_ms": round(max(tiempos), 2),
            "tiempo_minimo_ms": round(min(tiempos), 2),
            "total_tokens": total_tokens,
            "tasa_errores_pct": round((errores / len(self.registros)) * 100, 2),
            "consultas_por_tipo": tipos,
        }

logger.info("Clase RecolectorMetricas lista")

In [ ]:
# CELDA 4
# --- Base de conocimiento del taller de vulcanizacion ---
CONTEXTO_VULCANIZACION = """
Eres un asistente experto del taller de vulcanizacion y neumaticos.

SERVICIOS Y PRECIOS:
- Parche interno vulcanizado: $6.000
- Parche externo (temporal): $3.000
- Cambio de neumatico (c/u): $5.000
- Cambio de 4 neumaticos: $18.000
- Balanceo por rueda: $3.750
- Balanceo 4 ruedas: $15.000
- Alineacion computarizada: $12.000
- Alineacion + Balanceo 4 ruedas: $25.000
- Inflado con nitrogeno (c/u): $2.000
- Inflado con nitrogeno 4 ruedas: $7.000
- Revision de neumaticos (inspeccion): Gratis

TIEMPOS ESTIMADOS:
- Parche: 20-30 minutos
- Cambio de neumatico: 15-20 minutos por rueda
- Balanceo: 10-15 minutos por rueda
- Alineacion: 30-45 minutos

INFORMACION DEL TALLER:
- Horario: Lunes a Viernes 8:00-18:00, Sabado 9:00-14:00
- Se trabaja con todas las marcas de neumaticos
- Se recomienda balanceo cada 10.000 km
- Se recomienda alineacion cada 15.000 km o al notar el vehiculo jalando hacia un lado

INSTRUCCIONES:
- Responde solo con informacion disponible en este contexto
- Si no tienes la informacion, indica claramente que no esta disponible
- Siempre indica el precio cuando sea relevante
- Se cordial y profesional
"""

logger.info("Contexto RAG de vulcanizacion cargado")

In [ ]:
# CELDA 5
class AgenteObservable:
    """Agente de vulcanizacion con observabilidad completa."""

    def __init__(self, nombre: str):
        self.nombre = nombre
        self.metricas = RecolectorMetricas()
        self.modelo = "gpt-4.1"
        self.logger = logging.getLogger(f"agente.{nombre}")
        self.logger.setLevel(logging.DEBUG)
        if not self.logger.handlers:
            self.logger.addHandler(manejador_consola)
        self.client = OpenAI(
            api_key=os.getenv("GITHUB_TOKEN"),
            base_url=os.getenv("OPENAI_BASE_URL", "https://models.inference.ai.azure.com")
        )
        self.logger.info(f"Agente '{nombre}' iniciado con modelo {self.modelo}")

    def clasificar_consulta(self, mensaje: str) -> str:
        mensaje_lower = mensaje.lower()
        if any(p in mensaje_lower for p in ["parche", "pinchazo", "ponche"]):
            return "parche"
        elif any(p in mensaje_lower for p in ["cambio", "neumatico", "llanta", "goma"]):
            return "cambio_neumatico"
        elif "alineacion" in mensaje_lower or "alineación" in mensaje_lower:
            return "alineacion"
        elif "balanceo" in mensaje_lower:
            return "balanceo"
        elif "nitrogeno" in mensaje_lower or "nitrógeno" in mensaje_lower:
            return "nitrogeno"
        elif any(p in mensaje_lower for p in ["precio", "costo", "cuanto", "valor"]):
            return "consulta_precio"
        elif any(p in mensaje_lower for p in ["horario", "hora", "atienden"]):
            return "consulta_horario"
        else:
            return "consulta_general"

    def procesar(self, mensaje: str) -> str:
        """Procesa una consulta del cliente con observabilidad completa."""
        tipo_consulta = self.clasificar_consulta(mensaje)
        self.logger.info(f"Entrada [{tipo_consulta}]: {mensaje!r}")
        inicio = time.perf_counter()
        try:
            response = self.client.chat.completions.create(
                model=self.modelo,
                messages=[
                    {"role": "system", "content": CONTEXTO_VULCANIZACION},
                    {"role": "user", "content": mensaje}
                ],
                max_tokens=500,
                temperature=0.3
            )
            duracion_ms = (time.perf_counter() - inicio) * 1000
            tokens_in = response.usage.prompt_tokens
            tokens_out = response.usage.completion_tokens
            respuesta = response.choices[0].message.content
            self.metricas.registrar(duracion_ms, tokens_in, tokens_out,
                                    exitoso=True, consulta_tipo=tipo_consulta,
                                    modelo=self.modelo)
            self.logger.info(f"Salida [{tipo_consulta}] ({duracion_ms:.1f}ms, {tokens_in}+{tokens_out} tokens)")
            return respuesta
        except Exception as e:
            duracion_ms = (time.perf_counter() - inicio) * 1000
            self.logger.warning(f"Error al procesar [{tipo_consulta}]: {e}")
            self.metricas.registrar(duracion_ms, 0, 0,
                                    exitoso=False, consulta_tipo=tipo_consulta,
                                    modelo=self.modelo)
            return f"[ERROR] No se pudo procesar la consulta: {e}"

    def reporte(self):
        """Imprime un reporte estructurado de metricas."""
        resumen = self.metricas.resumen()
        self.logger.info("=== Reporte de Metricas ===")
        print(json.dumps(resumen, indent=2, ensure_ascii=False))

logger.info("Clase AgenteObservable lista")

In [ ]:
# CELDA 6
# --- Demostracion del sistema de vulcanizacion con observabilidad ---
print("=" * 60)
print("DEMO: Observabilidad en Agente de Vulcanizacion")
print("=" * 60)

agente = AgenteObservable("vulcanizadora-v1")

consultas = [
    "¿Cuánto cuesta un parche interno?",
    "Necesito cambiar los 4 neumáticos, ¿cuál es el precio?",
    "¿Hacen alineación computarizada?",
    "¿Cuál es el horario de atención?",
    "¿Cuánto cuesta la revisión técnica del auto?",  # fuera del contexto
]

for consulta in consultas:
    print(f"\n🔧 Cliente: {consulta}")
    print("-" * 50)
    resultado = agente.procesar(consulta)
    print(f"🔩 Vulcanizadora: {resultado}")

print("\n" + "=" * 60)
agente.reporte()

In [ ]:
# --- Ver todos los registros individuales ---
print("\n📋 REGISTROS INDIVIDUALES:")
print("=" * 60)
for i, r in enumerate(agente.metricas.registros, 1):
    estado = "OK" if r.exitoso else "ERROR"
    print(f"{i}. {estado} [{r.consulta_tipo}] "
          f"{r.tiempo_respuesta_ms}ms | "
          f"tokens: {r.tokens_entrada}↑ {r.tokens_salida}↓ | "
          f"{r.timestamp}")